In [0]:
%run ../configs/config

In [0]:
trans=f'{bronze_schema}.trans'
trans_df = spark.table(trans)

In [0]:
display(trans_df)

In [0]:
dropped_trans = (trans_df
                 .fillna({'k_symbol': 'Unknown', 'bank': 'Unknown', 'operation':'Unknown','account': '0'})
                 .dropDuplicates()
                 .withColumn('type', F.when(F.col('type') == 'PRIJEM', 'Credit')
                             .when(F.col('type') == 'VYDAJ', 'Debit')
                             .otherwise('Unknown'))
                 .withColumn('operation', 
                             F.when(F.col('operation') == 'PREVOD NA UCET','Transfer to Account' )
                             .when(F.col('operation') == 'PREVOD Z UCTU', 'Transfer to Account')
                             .otherwise('Unknown'))
                 .withColumnRenamed('k_symbol', 'payment_type')
                 .withColumn('payment_type',
                              F.when(F.col('payment_type') == 'SIPO', 'Household Payment')
                              .when(F.col('payment_type') == 'UVER', 'Loan Payment')
                              .when(F.col('payment_type') == ' ', 'Unknown')
                              .when(F.col('payment_type') == 'POJISTNE', 'Insurance Payment')
                              .when(F.col('payment_type') == 'DUCHOD', 'Pension Payment')
                              .otherwise('Unknown')
                              )
                 .withColumn('date', F.to_date(F.concat(F.lit('19'), F.col('date')
                                        .cast('string')), 'yyyyMMdd')
            )
)

In [0]:
display(dropped_trans)

In [0]:
print(f"Before drop: {trans_df.count()}")
print(f"After drop: {dropped_trans.count()}")

In [0]:
(
    dropped_trans
        .write
        .format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(f'{silver_schema}.transactions')
)